# Pilot analysis: Figure 3 up to 16 real images per class

Reads the classifier results written by `pilot_classifier.py` and draws the paper's Figure 3 layout
(mean accuracy vs real images per class, with and without 10 synthetic per real, with and without
image transforms) for the pilot generators. One seed, so no averaging across runs; error bars are the
Wald 95% CI on the 360-image held-out test set.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = "/workspace/synderm/pilot/results/results.csv"
FIG_OUT = "/workspace/synderm/pilot/results/figure3_pilot.png"


In [ ]:
results = pd.read_csv(RESULTS)
results = results.sort_values(["generator", "augmentation", "n_training_per_label"]).reset_index(drop=True)
results[["generator", "augmentation", "n_training_per_label", "n_train", "epochs_run", "top1_acc", "top1_ci_lower", "top1_ci_upper", "top3_acc"]]


## Accuracy table

Rows are real images per class; columns pair each generator with the real-only baseline.

In [ ]:
table = results.pivot_table(index=["augmentation", "n_training_per_label"], columns="generator", values="top1_acc")
table = (table * 100).round(1)
table


In [ ]:
gain = table.sub(table["none"], axis=0).drop(columns="none")
gain.columns = [f"gain vs real-only: {c}" for c in gain.columns]
gain


## Figure

Same layout as the paper's Figure 3: one column per generator, rows for with and without image
transforms. Solid line is real only, dashed is real plus 10 synthetic per real.

In [ ]:
generators = [g for g in results["generator"].unique() if g != "none"]
augs = ["aug_transforms", "None"]
aug_title = {"aug_transforms": "With image transforms", "None": "Without image transforms"}
colors = {"aug_transforms": "#6a4c9c", "None": "#e07a2f"}

fig, axes = plt.subplots(len(augs), len(generators), figsize=(4.2 * len(generators), 3.6 * len(augs)), squeeze=False, sharey=True)
for r, aug in enumerate(augs):
    for c, gen in enumerate(generators):
        ax = axes[r][c]
        base = results[(results.generator == "none") & (results.augmentation == aug)].sort_values("n_training_per_label")
        synth = results[(results.generator == gen) & (results.augmentation == aug)].sort_values("n_training_per_label")
        ax.errorbar(base.n_training_per_label, base.top1_acc * 100,
                    yerr=[(base.top1_acc - base.top1_ci_lower) * 100, (base.top1_ci_upper - base.top1_acc) * 100],
                    fmt="o-", color=colors[aug], capsize=3, label="real only")
        ax.errorbar(synth.n_training_per_label, synth.top1_acc * 100,
                    yerr=[(synth.top1_acc - synth.top1_ci_lower) * 100, (synth.top1_ci_upper - synth.top1_acc) * 100],
                    fmt="o--", color=colors[aug], capsize=3, label="real + 10 synthetic per real")
        ax.set_xscale("log", base=2); ax.set_xticks([1, 8, 16]); ax.set_xticklabels(["1", "8", "16"])
        ax.set_ylim(0, 60); ax.grid(alpha=0.3)
        if r == 0: ax.set_title(f"{gen} (edit)")
        if c == 0: ax.set_ylabel(f"{aug_title[aug]}\nMean accuracy (%)")
        if r == len(augs) - 1: ax.set_xlabel("Real training images per disease condition")
        ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Pilot: nine-way skin classification, one seed, 360-image held-out test", y=1.02)
fig.tight_layout()
fig.savefig(FIG_OUT, dpi=150, bbox_inches="tight")
print("saved", FIG_OUT)


## Refusals and top-ups per generator

How many synthetic slots were filled by top-up edits of another reference in the same class.

In [ ]:
topups = []
for pilot_dir in results["pilot_dir"].dropna().unique():
    syn = pd.read_csv(f"{pilot_dir}/synthetic.csv")
    topups.append({"pilot_dir": pilot_dir.split("/")[-1], "synthetic_rows": len(syn), "topups": int(syn.topup.sum()),
                   "refs_with_topups": int(syn[syn.topup].md5hash.nunique())})
pd.DataFrame(topups)
